In [ ]:
# ==============================================================
# [셀 1/6] 데이터 준비  (= model/LoopedTransformer/dataLT.py 와 같은 내용)
# ==============================================================
# 이 셀이 하는 일
#   CSV를 읽어서 모델이 쓸 수 있는 형태(입력 x, 정답 y)로 바꾼다.
#   여기서는 함수만 정의하고, 실제 호출은 [셀 5]의 main()에서 한다.
#
# 꼭 알아야 할 것
#   1) 입력 x = 7개
#        - 공정조건 4개   : P_laser, v_scan, d_laser, t_powder
#        - 파생 feature 3개: linear_energy(P/v), peclet_number(Pe), normalized_enthalpy(H*)
#   2) 정답 y = 3개: melt_width(W), aspect_ratio(R = 깊이/폭), melt_area(A)
#        - 깊이 D는 직접 예측하지 않고 R로 바꿔서 예측한다 (나중에 D = W * R 로 복원)
#        - balling은 같이 읽지만 학습에는 쓰지 않는다
#   3) 함수·클래스 4개가 이 순서로 쓰인다
#        load_316l_csv   : CSV 읽기 + 파생 feature, R 계산
#        make_iid_split  : train 70% / val 15% / test 15% 무작위 분할
#        fit_scalers     : 평균 0, 표준편차 1로 맞추는 기준을 train 데이터로만 계산
#        MeltPoolDataset : 샘플 하나를 {"x", "y", "balling", "id"} 형태로 꺼내주는 Dataset
#
# 다음 셀로 넘어가는 것: FEATURES, TARGETS, 위 함수·클래스
# ==============================================================

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


BASE_FEATURES = ["P_laser", "v_scan", "d_laser", "t_powder"]
DERIVED_FEATURES = ["linear_energy", "peclet_number", "normalized_enthalpy"]

# v3 physical-state representation:
#   W = melt-pool width [um]
#   R = D/W = melt-pool aspect ratio [-]
#   A = melt-pool cross-sectional area [um^2]
TARGETS = ["melt_width", "aspect_ratio", "melt_area"]
RISK = "balling"


def get_features(include_linear_energy: bool = True) -> list[str]:
    """Return model input features.

    Raw process variables are always retained. Linear energy density P/v is
    an additional derived feature; it does not replace P_laser or v_scan.
    """
    return BASE_FEATURES + (DERIVED_FEATURES if include_linear_energy else [])


FEATURES = get_features(True)


def load_316l_csv(path: str | Path) -> pd.DataFrame:
    """Load the supplied 316L single-track CSV and derive v3 geometry state."""
    path = Path(path)
    df = pd.read_csv(path, header=None, skiprows=1)
    if df.shape[1] != 9:
        raise ValueError(f"Expected 9 columns including ID, got {df.shape[1]}")

    df.columns = [
        "id", "P_laser", "v_scan", "d_laser", "t_powder",
        "melt_width", "melt_depth", "melt_area", "balling",
    ]
    df[df.columns] = df[df.columns].apply(pd.to_numeric, errors="raise")
    df["id"] = df["id"].astype(int)
    df["balling"] = df["balling"].astype(np.float32)

    # J/mm: W / (mm/s) = J/mm.
    df["linear_energy"] = df["P_laser"] / df["v_scan"]

    # v4 dimensionless thermal descriptors.
    # Conventions: d_laser is treated as the laser spot DIAMETER; v_scan is
    # converted from mm/s to m/s. These conventions match the normalized
    # enthalpy / Peclet definitions used by the scaling-law literature.
    #
    # Pe = v*d/alpha
    # H* = eta*P / (h_s * sqrt(alpha*d^3*v))
    # where h_s = rho*cp*(Tm-T0) is the sensible enthalpy density needed
    # to heat the material from T0 to the melting temperature.
    rho_316l = 7800.0              # kg/m^3
    cp_316l = 460.0                # J/(kg K)
    alpha_316l = 5.0e-6            # m^2/s, representative 316L value
    Tm_316l = 1678.0               # K
    T0_316l = 293.15               # K
    absorptivity_316l = 0.34
    h_s_vol = rho_316l * cp_316l * (Tm_316l - T0_316l)

    v_mps = np.maximum(df["v_scan"].to_numpy(dtype=float) / 1000.0, 1e-12)
    diameter_m = np.maximum(df["d_laser"].to_numpy(dtype=float) / 1000.0, 1e-12)
    power_w = np.maximum(df["P_laser"].to_numpy(dtype=float), 1e-12)
    df["peclet_number"] = v_mps * diameter_m / alpha_316l
    df["normalized_enthalpy"] = (
        absorptivity_316l * power_w
        / (h_s_vol * np.sqrt(alpha_316l * diameter_m**3 * v_mps))
    )

    # v3 physical representation.
    # Aspect ratio is intentionally computed from the measured geometry rather
    # than predicted independently:
    #     R = D / W
    # It will later be used to reconstruct physical depth as D = W * R.
    width = df["melt_width"].clip(lower=1e-6)
    df["aspect_ratio"] = df["melt_depth"] / width

    if not np.isfinite(df["aspect_ratio"]).all():
        raise ValueError("Non-finite aspect_ratio found in melt-pool data")

    return df


@dataclass
class DataScalers:
    x: StandardScaler
    y: StandardScaler
    features: list[str]
    targets: list[str]



def make_iid_split(
    df: pd.DataFrame,
    seed: int = 42,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Initial IID split. A process-space split can be added as a separate experiment."""
    train, temp = train_test_split(df, test_size=1.0 - train_ratio, random_state=seed)
    relative_val = val_ratio / (1.0 - train_ratio)
    val, test = train_test_split(temp, test_size=1.0 - relative_val, random_state=seed)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)



def fit_scalers(train_df: pd.DataFrame, include_linear_energy: bool = True) -> DataScalers:
    features = get_features(include_linear_energy)
    x = StandardScaler().fit(train_df[features])
    y = StandardScaler().fit(train_df[TARGETS])
    return DataScalers(x=x, y=y, features=features, targets=TARGETS.copy())


class MeltPoolDataset(Dataset):
    def __init__(self, df: pd.DataFrame, scalers: DataScalers):
        self.df = df.reset_index(drop=True)
        self.x = scalers.x.transform(self.df[scalers.features]).astype(np.float32)
        self.y = scalers.y.transform(self.df[scalers.targets]).astype(np.float32)
        self.balling = self.df[RISK].to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            "x": torch.from_numpy(self.x[idx]),
            "y": torch.from_numpy(self.y[idx]),
            "balling": torch.tensor(self.balling[idx]),
            "id": int(self.df.iloc[idx]["id"]),
        }


In [62]:
# ==============================================================
# [셀 2/6] Loss 함수  (= model/LoopedTransformer/lossLT.py)
# ==============================================================
# 이 셀이 하는 일
#   "모델이 얼마나 틀렸는지"를 숫자 하나로 계산하는 함수 2개를 정의한다.
#
# 꼭 알아야 할 것
#   1) DeepSupervisionLoss — 데이터 loss (실측값과 비교)
#        - 모델은 루프를 3번 돌며 예측을 3개 낸다 (y_hat_1, y_hat_2, y_hat_3)
#        - 마지막 예측만이 아니라 3개 모두 MSE로 채점한다 (= deep supervision)
#        - 뒤 단계일수록 가중치가 크다: gamma = 0.8 이면 0.64 : 0.8 : 1.0
#        - state_weights 로 W, R, A 중 무엇을 더 중요하게 볼지 조절 (기본 1 : 1 : 1)
#   2) PhysicsConsistencyLoss — 물리 loss (물리식과 비교)
#        - [셀 3] Physics Evaluator 가 낸 residual 을 제곱해서 평균
#        - confidence 가 있으면 가중 평균 (신뢰도가 낮은 항목은 덜 반영)
#   3) 최종 loss = 데이터 loss + alpha * 물리 loss   ([셀 5]에서 계산)
#        - 기본값 alpha = 0 → 물리 loss 는 계산·출력만 되고 학습에는 반영되지 않는다
# ==============================================================

from __future__ import annotations

import torch
from torch import nn


class DeepSupervisionLoss(nn.Module): # param 가중치 조절
    """
    L = L_data + alpha * L_physics
    """

    def __init__(self, gamma: float = 0.8, state_weights=(1.0, 1.0, 1.0)):
        super().__init__()
        weights = torch.as_tensor(state_weights, dtype=torch.float32)
        if weights.ndim != 1 or weights.numel() != 3:
            raise ValueError("state_weights must contain exactly 3 values: W, R, A")
        if torch.any(weights <= 0):
            raise ValueError("state_weights must be positive")
        self.gamma = gamma
        self.register_buffer("state_weights", weights)

    def forward(self, predictions, target):
        if target.size(-1) != self.state_weights.numel():
            raise ValueError(
                f"Expected target dimension {self.state_weights.numel()}, got {target.size(-1)}"
            )

        total = target.new_tensor(0.0)
        weight_sum = 0.0
        for k, pred in enumerate(predictions):
            loop_weight = self.gamma ** (len(predictions) - 1 - k)
            per_state = (pred - target).pow(2).mean(dim=0)
            state_loss = (per_state * self.state_weights.to(pred)).sum() / self.state_weights.sum()
            total = total + loop_weight * state_loss
            weight_sum += loop_weight
        return total / weight_sum

class PhysicsConsistencyLoss(nn.Module):
    """Mean squared physics residual"""

    def forward(self, feedbacks, confidences=None):
        if not feedbacks:
            return torch.tensor(0.0)

        terms = []
        for i, feedback in enumerate(feedbacks):
            if confidences is not None and confidences[i] is not None:
                mask = confidences[i]
                denom = mask.sum().clamp_min(1.0)
                terms.append((feedback.pow(2) * mask).sum() / denom)
            else:
                terms.append(feedback.pow(2).mean())
        return torch.stack(terms).mean()


In [63]:
# ==============================================================
# [셀 3/6] Physics Evaluator  (= model/LoopedTransformer/physicsLT.py)
# ==============================================================
# 이 셀이 하는 일
#   모델의 예측값이 물리식 및 기하 일관성과 얼마나 어긋나는지(residual)를 계산한다.
#   이 residual 이 [셀 4] 모델의 루프로 다시 들어간다.
#
# 꼭 알아야 할 것
#   1) 클래스 3개 중 실제로 쓰는 것은 RosenthalPhysicsEvaluator
#        - ZeroPhysicsEvaluator   : residual 을 항상 0으로 반환 (구조 검증용, --physics none)
#        - DeltaFeedbackEvaluator : 디버그용 (현재 미사용)
#   2) evaluate(x, 예측값) 동작 순서
#        a. 표준화된 x, 예측값을 원래 단위로 되돌린다 (_physical_x, _physical_y)
#        b. 물리식으로 기준값 계산
#             rosenthal_width : P, v 로 폭 W 추정
#             scaling_depth   : H*, Pe 로 깊이 D 추정 → 기준 R = scaling D / 예측 W
#        c. residual 3개 계산
#             r_width  = (예측 W - Rosenthal W) / Rosenthal W
#             r_aspect = 0.25 * (log 예측 R - log 기준 R)
#             r_area   = (예측 A - pi/4*예측W*예측D) / (pi/4*예측W*예측D)
#                        예측D = 예측W * 예측R. 물리식이 아니라 예측값 3개끼리 앞뒤가 맞는지 검사
#        d. residual 을 -5 ~ +5 로 자른다 (값 폭주 방지)
#        e. confidence: 예측 R 이 0.2 미만 또는 2.0 초과면 R, A 항목 신뢰도를 0.1배로 낮춘다
#        ※ residual 계산에 실측값은 쓰지 않는다 (공정조건 + 예측값만 사용).
#          실측값과의 비교는 [셀 2] 데이터 loss 가 담당한다
#   3) 주의
#        - x 의 0, 1, 2번째 열을 P, v, d 로 가정한다 → FEATURES 순서를 바꾸면 안 된다
#        - 물성값(흡수율 0.34, 밀도 7800 등)은 316L 기준 고정값
# ==============================================================

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass

import math
import torch


@dataclass
class PhysicsFeedback:
    residual: torch.Tensor
    confidence: torch.Tensor | None = None


class PhysicsEvaluator(ABC):
    @abstractmethod
    def evaluate(self, process_x: torch.Tensor, predicted_state: torch.Tensor) -> PhysicsFeedback:
        raise NotImplementedError


class ZeroPhysicsEvaluator(PhysicsEvaluator):
    """아키텍처 검증 전용"""

    def evaluate(self, process_x, predicted_state):
        return PhysicsFeedback(residual=torch.zeros_like(predicted_state))


class RosenthalPhysicsEvaluator(PhysicsEvaluator): #Rosenthal의 열역학 모델
    """
    물리 모델:
        W = sqrt( 8/(pi*e) * eta*P / (rho*Cp*v*DeltaT) )
        R = D/W
        A = pi/4 * W * D

    평가:
        L=λW*LW​+λD​*LD​+λA*​LA​
    """

    def __init__(
        self,
        x_mean,
        x_scale,
        y_mean,
        y_scale,
        absorptivity: float = 0.34,
        density: float = 7800.0,
        heat_capacity: float = 460.0,
        thermal_diffusivity: float = 5.0e-6,
        melting_temperature: float = 1678.0,
        preheat_temperature: float = 293.15,
        scaling_k: float = 0.60,
        residual_clip: float = 5.0,
    ):
        self.x_mean = torch.as_tensor(x_mean, dtype=torch.float32)
        self.x_scale = torch.as_tensor(x_scale, dtype=torch.float32)
        self.y_mean = torch.as_tensor(y_mean, dtype=torch.float32)
        self.y_scale = torch.as_tensor(y_scale, dtype=torch.float32)
        self.absorptivity = absorptivity
        self.density = density
        self.heat_capacity = heat_capacity
        self.thermal_diffusivity = thermal_diffusivity
        self.melting_temperature = melting_temperature
        self.preheat_temperature = preheat_temperature
        self.scaling_k = scaling_k
        self.residual_clip = residual_clip

    def _physical_x(self, x: torch.Tensor) -> torch.Tensor:
        mean = self.x_mean.to(device=x.device, dtype=x.dtype)
        scale = self.x_scale.to(device=x.device, dtype=x.dtype)
        return x * scale + mean

    def _physical_y(self, y: torch.Tensor) -> torch.Tensor:
        mean = self.y_mean.to(device=y.device, dtype=y.dtype)
        scale = self.y_scale.to(device=y.device, dtype=y.dtype)
        return y * scale + mean

    def thermal_features(self, process_x: torch.Tensor):
        x = self._physical_x(process_x)
        P = x[:, 0].clamp_min(1e-6)
        v = (x[:, 1] / 1000.0).clamp_min(1e-9)
        diameter = (x[:, 2] / 1000.0).clamp_min(1e-9)

        Pe = v * diameter / self.thermal_diffusivity
        hs_vol = self.density * self.heat_capacity * (
            self.melting_temperature - self.preheat_temperature
        )
        h_star = self.absorptivity * P / (
            hs_vol * torch.sqrt(self.thermal_diffusivity * diameter**3 * v)
        )
        return P, v, diameter, Pe, h_star

    def rosenthal_width(self, process_x: torch.Tensor) -> torch.Tensor:
        """P 및 v 원시 입력을 사용하여 로젠탈 폭을 마이크로미터 단위로 반환합니다."""
        P, v, _, _, _ = self.thermal_features(process_x)
        delta_T = self.melting_temperature - self.preheat_temperature
        width_m = torch.sqrt(
            (8.0 / (math.pi * math.e))
            * self.absorptivity * P
            / (self.density * self.heat_capacity * v * delta_T)
        )
        return width_m * 1e6

    def scaling_depth(self, process_x: torch.Tensor) -> torch.Tensor:
        """마이크로미터 단위의 soft 스케일링 depth 추정치를 반환합니다.

        Using beta = H*/pi and L_th = Pe^(-1/2),
            D / d_laser ~= K * beta * L_th.

        분석용이지 Target은 아님에 주의.
        """
        _, _, diameter, Pe, h_star = self.thermal_features(process_x)
        beta = h_star / math.pi
        normalized_depth = self.scaling_k * beta / torch.sqrt(Pe.clamp_min(1e-12))
        return diameter * normalized_depth * 1e6

    def evaluate(self, process_x: torch.Tensor, predicted_state: torch.Tensor) -> PhysicsFeedback:
        pred = self._physical_y(predicted_state)
        pred_w, pred_r, pred_a = pred.unbind(dim=-1)
        eps = torch.tensor(1e-6, device=pred.device, dtype=pred.dtype)

        ros_w = self.rosenthal_width(process_x)
        phys_d = self.scaling_depth(process_x)
        phys_r = phys_d / pred_w.clamp_min(eps)
        phys_r = phys_r.clamp(min=0.15, max=2.5)
        pred_d = pred_w * pred_r

        r_width = (pred_w - ros_w) / (ros_w.abs() + eps)

        # 로그-비 잔차는 큰 종횡비 값이 피드백을 지배하는 것을 방지하고
        # 기하학적 오차를 대칭적으로 처리합니다.
        raw_r_residual = (
            torch.log(pred_r.clamp_min(1e-4))
            - torch.log(phys_r.clamp_min(1e-4))
        )

        # 분석 법칙이 모델 학습을 압도하지 않도록 기하 신뢰도를 조절합니다.
        R_PHYSICS_CONFIDENCE = 0.25
        r_aspect = R_PHYSICS_CONFIDENCE * raw_r_residual

        # 타원 면적 일관성 식: A ≈ pi/4 * W^2 * R (W * D)
        ellipse_area = math.pi / 4.0 * pred_w.clamp_min(eps) * pred_d.clamp_min(eps)
        r_area = (pred_a - ellipse_area) / (ellipse_area.abs() + eps)

        residual = torch.stack([r_width, r_aspect, r_area], dim=-1)
        residual = torch.clamp(residual, -self.residual_clip, self.residual_clip)

        # Balling 또는 Extreme R 상태(예: R < 0.2 또는 R > 2.0)에서는 단순 타원 가정의 신뢰도를 감쇄합니다.
        regime_penalty = torch.ones_like(pred_r)
        extreme_mask = (pred_r < 0.2) | (pred_r > 2.0)
        regime_penalty[extreme_mask] = 0.1

        # R_PHYSICS_CONFIDENCE 및 면적 일관성 신뢰도에 물리적 영역 패널티를 곱해줍니다.
        confidence_r_tensor = torch.full_like(r_width, R_PHYSICS_CONFIDENCE) * regime_penalty
        confidence_area = torch.ones_like(r_area) * regime_penalty

        confidence = torch.stack(
            [torch.ones_like(r_width), confidence_r_tensor, confidence_area], dim=-1
        )
        return PhysicsFeedback(residual=residual, confidence=confidence)


class DeltaFeedbackEvaluator(PhysicsEvaluator):
    """Debug 전용"""

    def __init__(self, state_dim: int = 3):
        self.state_dim = state_dim
        self.previous = None

    def reset(self):
        self.previous = None

    def evaluate(self, process_x, predicted_state):
        if self.previous is None:
            residual = torch.zeros_like(predicted_state)
        else:
            residual = predicted_state - self.previous
        self.previous = predicted_state.detach()
        return PhysicsFeedback(residual=residual)


In [64]:
# ==============================================================
# [셀 4/6] Looped Transformer 모델  (= model/LoopedTransformer/modelLT.py)
# ==============================================================
# 이 셀이 하는 일
#   "예측 → 물리 평가 → 보정"을 여러 번 반복하는 모델 구조를 정의한다.
#
# 꼭 알아야 할 것
#   1) 일반 Transformer 와 다른 점: sequence 축이 "시간"이 아니라 "반복 횟수(loop)"
#   2) forward(x) 흐름 (기본 3회 반복)
#        a. FeatureTokenizer: 숫자 7개를 각각 64차원 벡터(토큰)로 변환 → [B, 7, 64]
#        b. 맨 뒤에 state token 1개를 붙임 → [B, 8, 64]
#             state token = 모델의 "현재 추론 상태"를 담는 메모. 루프마다 갱신된다
#        c. TransformerEncoder (2층, head 4개) 통과
#        d. state token 자리의 출력 → state_head → 예측 [W, R, A]
#        e. physics_evaluator 로 residual 계산 (넘기지 않으면 0)
#        f. residual → feedback_encoder → state_update → state token 갱신
#             state = state + 0.25 * (갱신량)   ← 0.25 로 한 번에 크게 바뀌지 않게 조절
#        g. a~f 를 loop_steps 번 반복하고, 매 단계 예측을 모두 모아 반환 (LoopOutput)
#   3) StateMLP: 비교용 baseline 모델 (이 노트북의 학습에서는 사용하지 않음)
# ==============================================================

# 2026. 601Solutions
# 유의해야할 점: 기존 Transformer 모델과 달리 이 Proj의 Transformer의 sequence는 데이터의 시간축이 아니라 추론 iteration이다.

"""
Mathmatical Modeling
x=[P,v,d,t,E_L,Pe,H*]   (공정조건 4 + 파생 feature 3)
y=[W,R,A]               (R = D/W)

x→fθ​(x)→y^​0​
r0​=PhysicsEvaluator(x,y^​0​)

h1​=Update(h0​,r0​)
^y​1​=fθ​(x,h1​)

r1​=PhysicsEvaluator(x,y^​1​)
Loop...
"""

from __future__ import annotations

from dataclasses import dataclass

import torch
from torch import nn


@dataclass
class LoopOutput:
    predictions: list[torch.Tensor]
    feedbacks: list[torch.Tensor]
    confidences: list[torch.Tensor | None] | None = None


class FeatureTokenizer(nn.Module):
    """
    스칼라값 to 토큰

    Input:
        [B, n_features]   (현재 7)

    Output:
        [B, n_features, d_model]
    """

    def __init__(self, n_features: int, d_model: int):
        super().__init__()
        self.scalar_projection = nn.Linear(1, d_model)
        self.feature_embedding = nn.Parameter(torch.randn(n_features, d_model) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.scalar_projection(x.unsqueeze(-1))
        return tokens + self.feature_embedding.unsqueeze(0)

def make_encoder(d_model: int, n_heads: int, n_layers: int, dropout: float):
    layer = nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=n_heads,
        dim_feedforward=4 * d_model,
        dropout=dropout,
        activation="gelu",
        batch_first=True,
        norm_first=True,
    )
    return nn.TransformerEncoder(layer, num_layers=n_layers)


class LoopedTransformer(nn.Module): # L/T Main

    def __init__(
        self,
        n_features: int = 5,
        state_dim: int = 3,
        feedback_dim: int = 3,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.1,
        loop_steps: int = 3,
    ):
        super().__init__()
        self.state_dim = state_dim
        self.feedback_dim = feedback_dim
        self.d_model = d_model
        self.loop_steps = loop_steps

        self.tokenizer = FeatureTokenizer(n_features, d_model)
        self.state_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = make_encoder(d_model, n_heads, n_layers, dropout)

        self.state_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, state_dim),
        )

        self.feedback_encoder = nn.Sequential(
            nn.Linear(feedback_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.state_update = nn.Sequential(
            nn.LayerNorm(2 * d_model),
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(
        self,
        x: torch.Tensor,
        physics_evaluator=None,
        loop_steps: int | None = None,
    ) -> LoopOutput:
        bsz = x.size(0)
        steps = loop_steps or self.loop_steps

        process_tokens = self.tokenizer(x)
        state = self.state_token.expand(bsz, -1, -1)

        predictions = []
        feedbacks = []
        confidences = []

        for _ in range(steps):
            tokens = torch.cat([process_tokens, state], dim=1)
            encoded = self.transformer(tokens)

            state_hidden = encoded[:, -1, :]
            prediction = self.state_head(state_hidden)
            predictions.append(prediction)

            if physics_evaluator is None:
                feedback = torch.zeros_like(prediction)
                confidence = None
            else:
                result = physics_evaluator.evaluate(x, prediction)
                feedback = result.residual
                confidence = result.confidence

            feedbacks.append(feedback)
            confidences.append(confidence)

            feedback_hidden = self.feedback_encoder(feedback)
            # 주요 수정 사항.
            PHYSICS_FEEDBACK_SCALE = 0.25
            state = state + PHYSICS_FEEDBACK_SCALE * (self.state_update(
                torch.cat([state_hidden, feedback_hidden], dim=-1)
            )).unsqueeze(1)

        return LoopOutput(predictions=predictions, feedbacks=feedbacks, confidences=confidences)


class StateMLP(nn.Module):
    """baseline."""

    def __init__(self, n_features=4, state_dim=3, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, state_dim),
        )

    def forward(self, x):
        return self.net(x)




In [65]:
# ==============================================================
# [셀 5/6] 학습 실행
# ==============================================================
# 이 셀이 하는 일
#   셀 1~4 에서 정의한 것들을 조립해서 학습하고, 가장 좋은 모델을 .pt 파일로 저장한다.
#
# 꼭 알아야 할 것
#   1) 주요 설정 기본값 (노트북에서는 명령행 인자를 줄 수 없으므로 모두 기본값으로 실행)
#        --data /content/316L_meltpool.csv (Colab 경로)   --epochs 100   --batch-size 64
#        --lr 1e-3   --loop-steps 3   --physics rosenthal   --physics-alpha 0.0
#        --patience 20 (20 epoch 동안 개선이 없으면 조기 종료)
#   2) main() 흐름
#        a. 데이터 로드 → 분할 → 스케일러 → DataLoader            ([셀 1])
#        b. 모델 생성 + RosenthalPhysicsEvaluator 생성           ([셀 4], [셀 3])
#             Evaluator 에 스케일러 평균·표준편차를 넘겨 원래 단위로 되돌릴 수 있게 한다
#        c. epoch 반복
#             train: 예측 → loss([셀 2]) → 역전파 → gradient clip(1.0) → AdamW 업데이트
#             val  : 같은 계산을 학습 없이 수행해 성능만 측정
#        d. val 데이터 loss 가 최저를 갱신할 때마다 체크포인트 저장
#             저장 내용: 가중치, x/y 스케일러, feature·target 목록, 설정값
#        e. 1, 10, 20, ... epoch 마다 loss 출력
#   3) 출력 읽는 법: train / val = 전체 loss, data_val = 데이터 loss, physics_val = 물리 loss
#        (alpha = 0 이라 val 과 data_val 이 항상 같다)
#   4) 주의
#        - test_df 는 만들기만 하고 이 노트북에서 평가하지 않는다
#        - --model single 을 줘도 LoopedTransformer 가 만들어진다 (physics 만 0 으로 바뀜)
#        - --physics-feedback-scale, --physics-r-confidence 는 받기만 하고 쓰이지 않는다
#          (실제 값은 [셀 4], [셀 3] 코드 안에 0.25 로 고정)
# ==============================================================

from __future__ import annotations

import argparse
from pathlib import Path

import torch
from torch.utils.data import DataLoader

def build_model(args, n_features: int):
    common = dict(
        n_features=n_features,
        d_model=args.d_model,
        n_heads=args.n_heads,
        n_layers=args.n_layers,
        dropout=args.dropout,
    )
    return LoopedTransformer(loop_steps=args.loop_steps, **common)


def parse_state_weights(values):
    if len(values) != 3:
        raise ValueError("--state-weights requires exactly 3 values: W R A")
    weights = tuple(float(v) for v in values)
    if any(v <= 0 for v in weights):
        raise ValueError("All state weights must be positive")
    return weights


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="C:/Users/wjsgu/OneDrive/Desktop/2026/경남AI·SW경진대회/aero-twin-agent/data/316L_meltpool.csv")
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--loop-steps", type=int, default=3)
    parser.add_argument("--model", choices=["single", "looped"], default="looped")
    parser.add_argument("--physics", choices=["none", "rosenthal"], default="rosenthal")
    parser.add_argument("--physics-alpha", type=float, default=0.0)
    parser.add_argument("--linear-energy", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--state-weights", nargs=3, default=("1.0", "1.0", "1.0"), metavar=("W", "R", "A"))
    parser.add_argument("--gamma", type=float, default=0.8)
    parser.add_argument("--patience", type=int, default=20)
    parser.add_argument("--selection", choices=["data", "total"], default="data")
    parser.add_argument("--physics-feedback-scale",type=float,default=0.25)
    parser.add_argument("--physics-r-confidence", type=float, default=0.25)
    parser.add_argument("--d-model", type=int, default=64)
    parser.add_argument("--n-heads", type=int, default=4)
    parser.add_argument("--n-layers", type=int, default=2)
    parser.add_argument("--dropout", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--out", default="looped_transformer_best.pt")
    args, unknown = parser.parse_known_args()
    args.state_weights = parse_state_weights(args.state_weights)

    torch.manual_seed(args.seed)
    df = load_316l_csv(args.data)
    train_df, val_df, test_df = make_iid_split(df, seed=args.seed)
    scalers = fit_scalers(train_df, include_linear_energy=args.linear_energy)

    train_loader = DataLoader(
        MeltPoolDataset(train_df, scalers), batch_size=args.batch_size, shuffle=True
    )
    val_loader = DataLoader(MeltPoolDataset(val_df, scalers), batch_size=args.batch_size)

    model = build_model(args, n_features=len(scalers.features))
    if args.physics == "rosenthal" and args.model == "looped":
        physics = RosenthalPhysicsEvaluator(
            x_mean=scalers.x.mean_, x_scale=scalers.x.scale_,
            y_mean=scalers.y.mean_, y_scale=scalers.y.scale_,
        )
    else:
        physics = ZeroPhysicsEvaluator()

    data_loss_fn = DeepSupervisionLoss(gamma=args.gamma, state_weights=args.state_weights)
    physics_loss_fn = PhysicsConsistencyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

    best_metric = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    out_path = Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, args.epochs + 1):
        model.train()
        train_total = train_data = train_phys = 0.0

        for batch in train_loader:
            optimizer.zero_grad()
            out = model(batch["x"], physics_evaluator=physics)
            data_loss = data_loss_fn(out.predictions, batch["y"])
            phys_loss = physics_loss_fn(out.feedbacks, out.confidences)
            loss = data_loss + args.physics_alpha * phys_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            n = batch["x"].size(0)
            train_total += loss.item() * n
            train_data += data_loss.item() * n
            train_phys += phys_loss.item() * n

        train_total /= len(train_loader.dataset)
        train_data /= len(train_loader.dataset)
        train_phys /= len(train_loader.dataset)

        model.eval()
        val_total = val_data = val_phys = 0.0
        with torch.no_grad():
            for batch in val_loader:
                out = model(batch["x"], physics_evaluator=physics)
                data_loss = data_loss_fn(out.predictions, batch["y"])
                phys_loss = physics_loss_fn(out.feedbacks, out.confidences)
                loss = data_loss + args.physics_alpha * phys_loss
                n = batch["x"].size(0)
                val_total += loss.item() * n
                val_data += data_loss.item() * n
                val_phys += phys_loss.item() * n

        val_total /= len(val_loader.dataset)
        val_data /= len(val_loader.dataset)
        val_phys /= len(val_loader.dataset)

        selection_metric = val_data if args.selection == "data" else val_total
        if selection_metric < best_metric:
            best_metric = selection_metric
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "model": model.state_dict(),
                    "x_scaler": scalers.x,
                    "y_scaler": scalers.y,
                    "features": scalers.features,
                    "targets": scalers.targets,
                    "config": vars(args),
                    "state_representation": "[melt_width, aspect_ratio=D/W, melt_area]",
                },
                out_path,
            )
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:03d} train={train_total:.5f} "
                f"val={val_total:.5f} data_val={val_data:.5f} physics_val={val_phys:.5f}"
            )

        if args.patience > 0 and epochs_without_improvement >= args.patience:
            print(f"Early stopping at epoch {epoch} (patience={args.patience}).")
            break

    print("Training complete.")
    print(f"Best validation {args.selection} objective: {best_metric:.5f} at epoch {best_epoch}")
    print(f"model={args.model}, physics={args.physics}, physics_alpha={args.physics_alpha}")
    print(f"state_representation=[W, R=D/W, A], state_weights={args.state_weights}")
    print(f"features={scalers.features}")
    print("NOTE: Rosenthal is a soft analytical consistency model, not ground truth.")


if __name__ == "__main__":
    main()



/tmp/ipykernel_738/63667794.py:64: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=n_layers)


epoch=001 train=1.12062 val=0.89447 data_val=0.89447 physics_val=0.05792
epoch=010 train=0.45240 val=0.44047 data_val=0.44047 physics_val=0.03795
epoch=020 train=0.20561 val=0.17718 data_val=0.17718 physics_val=0.04287
epoch=030 train=0.17755 val=0.16673 data_val=0.16673 physics_val=0.06453
epoch=040 train=0.16846 val=0.19555 data_val=0.19555 physics_val=0.05393
epoch=050 train=0.15537 val=0.14003 data_val=0.14003 physics_val=0.06096
epoch=060 train=0.13242 val=0.14336 data_val=0.14336 physics_val=0.08311
epoch=070 train=0.13398 val=0.13894 data_val=0.13894 physics_val=0.03421
epoch=080 train=0.13609 val=0.11979 data_val=0.11979 physics_val=0.03923
epoch=090 train=0.12417 val=0.12237 data_val=0.12237 physics_val=0.06185
epoch=100 train=0.11607 val=0.12911 data_val=0.12911 physics_val=0.05539
Early stopping at epoch 100 (patience=20).
Training complete.
Best validation data objective: 0.11979 at epoch 80
model=looped, physics=rosenthal, physics_alpha=0.0
state_representation=[W, R=D/W, 

In [66]:
# ==============================================================
# [셀 6/6] 예측 결과 확인
# ==============================================================
# 이 셀이 하는 일
#   저장된 체크포인트로 validation 데이터를 예측하고, 실측값과 나란히 표로 비교한다.
#   (interfaceLT.py 의 getPredict 와 거의 같은 과정)
#
# 꼭 알아야 할 것
#   1) 흐름: 체크포인트 로드 → 같은 seed(42)로 다시 분할 → 스케일러 복원 → 모델 생성·가중치 적용
#            → val 입력 표준화 → 예측 → 원래 단위로 역변환 → 실측/예측/오차 표 생성
#   2) 표 컬럼: Actual_* (실측), Pred_* (예측), Error_* (실측 - 예측)
#        aspect_ratio 는 깊이/폭 비율이므로, 깊이 오차를 보려면 W * R 로 환산해야 한다
#   3) 주의
#        - model(x_tensor) 에 physics_evaluator 를 넘기지 않아 residual 이 0 → 학습 때와 조건이 다르다
#        - 주석은 "상위 10개"지만 head(10) 이 없어 val 전체를 표시한다
#        - 평가 대상이 val 이다. val 은 [셀 5]에서 best 모델을 고르는 데 쓰였으므로
#          성능이 실제보다 약간 좋게 보일 수 있다 (test_df 로 평가하는 것이 원칙)
# ==============================================================

import torch
import pandas as pd
import numpy as np

# 1. 저장된 체크포인트 로드 (weights_only=False 추가하여 StandardScaler 로드 허용)
checkpoint_path = "looped_transformer_best.pt"
checkpoint = torch.load(checkpoint_path, weights_only=False)

# 2. 데이터셋 로드 및 분할
df = load_316l_csv("C:/Users/wjsgu/OneDrive/Desktop/2026/경남AI·SW경진대회/aero-twin-agent/data/316L_meltpool.csv")
train_df, val_df, test_df = make_iid_split(df, seed=42)

# 저장되어 있던 Scaler 및 가중치 복원
scaler_x = checkpoint["x_scaler"]
scaler_y = checkpoint["y_scaler"]

# 3. 모델 초기화 및 가중치 적용
model = LoopedTransformer(n_features=len(FEATURES), loop_steps=3)
model.load_state_dict(checkpoint["model"])
model.eval()

# 4. 검증 데이터 준비
# 수동으로 스케일링된 x 텐서 생성
x_scaled = scaler_x.transform(val_df[FEATURES]).astype(np.float32)
x_tensor = torch.from_numpy(x_scaled)

# 5. 예측 수행
with torch.no_grad():
    output = model(x_tensor)
    # LoopedTransformer의 최종 루프 스텝(index: -1) 예측값 선택
    final_predictions_scaled = output.predictions[-1].numpy()

# 6. 역변환 (Inverse Transform)을 통한 원래 스케일 복원
final_predictions = scaler_y.inverse_transform(final_predictions_scaled)

# 7. 실제값과 예측값 비교용 DataFrame 생성
comparison_df = pd.DataFrame()
for i, target_col in enumerate(TARGETS):
    comparison_df[f"Actual_{target_col}"] = val_df[target_col]
    comparison_df[f"Pred_{target_col}"] = final_predictions[:, i]
    comparison_df[f"Error_{target_col}"] = comparison_df[f"Actual_{target_col}"] - comparison_df[f"Pred_{target_col}"]

# 상위 10개 결과 출력
print("--- Validation Set Predictions vs Ground Truth (First 10 Rows) ---")
display(comparison_df.round(4))


--- Validation Set Predictions vs Ground Truth (First 10 Rows) ---


/tmp/ipykernel_738/63667794.py:64: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=n_layers)


,Actual_melt_width,Pred_melt_width,Error_melt_width,Actual_aspect_ratio,Pred_aspect_ratio,Error_aspect_ratio,Actual_melt_area,Pred_melt_area,Error_melt_area
0,117.300,142.464401,-25.1644,0.8244,0.4776,0.3468,9203.400,10015.933594,-812.5336
1,232.600,253.079895,-20.4799,1.1363,1.1995,-0.0632,56199.400,57365.562500,-1166.1625
2,85.900,112.737503,-26.8375,0.6659,0.6276,0.0383,4268.700,6557.620117,-2288.9201
3,111.900,116.192398,-4.2924,2.0232,2.1551,-0.1318,20580.300,23938.453125,-3358.1531
4,109.651,130.058807,-20.4079,0.9866,0.8973,0.0892,8134.272,12122.468750,-3988.1968
...,...,...,...,...,...,...,...,...,...
96,148.100,132.999496,15.1005,0.2957,0.4789,-0.1831,8086.500,8119.492676,-32.9932
97,114.751,145.089005,-30.3380,0.5152,0.4861,0.0292,9992.565,10510.042969,-517.4780
98,235.622,201.755798,33.8662,0.0720,0.2583,-0.1862,10091.447,11365.772461,-1274.3255
99,117.500,124.374901,-6.8749,1.0800,1.1591,-0.0791,13692.200,14822.830078,-1130.6301
